In [ ]:
import os
import torch
from l5kit.configs import load_config_data

# Config 파일 로드
CONFIG_PATH = 'config.yaml'
cfg = load_config_data(CONFIG_PATH)

# 주요 학습 파라미터 설정
train_batch_size = cfg['train_data_loader']['batch_size']
train_shuffle = cfg['train_data_loader']['shuffle']
num_workers = cfg['train_data_loader']['num_workers']
max_num_steps = cfg['train_params']['max_num_steps']
checkpoint_every_n_steps = cfg['train_params']['checkpoint_every_n_steps']
eval_every_n_steps = cfg['train_params']['eval_every_n_steps']
print(f'Max Steps: {max_num_steps}, Checkpoint every {checkpoint_every_n_steps} steps')


In [ ]:
# 1. 데이터 준비
# LocalDataManager을 사용해 데이터 불러오고 EgoDataset을 활용해 학습용 데이터 구성

# build_rasterizer사용해 차량 주변 환경->이미지로 변환
# DataLoader 통해 batch 단위로 데이터 로드
# -병렬 연산(GPU 사용)을 하기 위해 batch단위로 데이터 보내서 여러 sample 동시 처리(연산 속도 증가)
# -평균적 경향 반영하며 모델 일반화(batch size=16~128)

In [ ]:
import os
import torch
from l5kit.configs import load_config_data

# Config 파일 로드
CONFIG_PATH = 'config.yaml'
cfg = load_config_data(CONFIG_PATH)

# 주요 학습 파라미터 설정
train_batch_size = cfg['train_data_loader']['batch_size']
train_shuffle = cfg['train_data_loader']['shuffle']
num_workers = cfg['train_data_loader']['num_workers']
max_num_steps = cfg['train_params']['max_num_steps']
checkpoint_every_n_steps = cfg['train_params']['checkpoint_every_n_steps']
eval_every_n_steps = cfg['train_params']['eval_every_n_steps']
print(f'Max Steps: {max_num_steps}, Checkpoint every {checkpoint_every_n_steps} steps')


In [ ]:
# 2. Resnet 모델 구축

# EgoResNetModel이 Resnet backbone으로 사용 (기본적인 특징 추출 신경망) 
# Resnet이 CNN역할 하면서 입력 데이터를 저차원 특징 맵으로 변환함
# 출력 크기 2로 설정->예측값 생성
# forward()함수에서 특징 추출
# Linear layer로 최종 예측

In [ ]:
# backbone=이미지에서 의미 있는 features 추출하는 신경망
# 이미지 입력->backbone->추가 layer(예측기)->출력

In [ ]:
#Resnet 기반 모델 정의
from l5kit.planning.rasterized.model import RasterizedPlanningModel

model = RasterizedPlanningModel(
        model_arch="resnet50",
        num_input_channels=rasterizer.num_channels(),
        num_targets=3 * cfg["model_params"]["future_num_frames"], 
        weights_scaling= [1., 1., 1.],
        criterion=nn.MSELoss(reduction="none")
        )
print(model)


In [ ]:
# 3. 모델 학습
# MSELoss를 손실함수로 설정->예측값과 실제값 차이 최소화
# -예측값이 연속적인 좌표(output_size=2)가 x,y로 이루어진 연속적 위치 좌표
# -작은 오차의 미세 조정이 가능함(오차가 커질수록 패널티를 주기 때문에 정확한 좌표 예측하도록 유도 가능)
# Adam optimizer를 사용해 가중치 업데이트
# -SGD변형이라 일반 SGD보다 학습 속도 빠름
# -각 가중치마다 다른 learning rate 자동 조정 (초반에서 빠르게 학습하고 후반에 미세조정)
# -비정형, 노이즈 있는 데이터에서도 안정적으로 최적화 가능
## (찾아보니 진동이 적고 빠르게 수렴해서 "실전 모델 학습에서 많이 쓴다"는 말이 있는데 맞는지는 잘 모르겠어서 ##로 적습니다.)

In [ ]:
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model.to(device)
tr_it = iter(train_loader)

progress_bar = tqdm(range(cfg["train_params"]["max_num_steps"]))
optimizer = optim.Adam(model.parameters(), lr=1e-3)
losses_train = []
model.train()
torch.set_grad_enabled(True)

for _ in progress_bar:
    try:
        data = next(tr_it)
    except StopIteration:
        tr_it = iter(train_loader)
        data = next(tr_it)
        
    data = {k: v.to(device) for k, v in data.items()}
    result = model(data)
    print(result)
    
    loss = result["loss"]
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses_train.append(loss.item())
    progress_bar.set_description(f"loss: {loss.item()} loss(avg): {np.mean(losses_train)}")
plt.plot(np.arange(len(losses_train)), losses_train, label="train loss")
plt.xlabel('step')
plt.ylabel('loss')
plt.legend()
plt.show()

In [ ]:
model.eval()
total_val_loss = 0

with torch.no_grad():
    for batch in val_dataloader:
        data = batch["image"].to(device)
        targets = batch["target_positions"].to(device)

        outputs = model(data)
        loss = criterion(outputs, targets)
        total_val_loss += loss.item()

    print(f"Validation Loss: {total_val_loss / len(val_dataloader)}")


In [ ]:
# # Lossfuncion 및 optimizer 설정 (최적화)
# criterion = nn.MSELoss()                                    # 예측 경로-실제 경로 차이 최소화 위해 MSELoss 사용
# optimizer = optim.Adam(model.parameters(), lr=1e-4)         # Adam optimizer

# # 학습 루프
# num_epochs = 10                                             # 학습 반복 횟수
# for epoch in range(num_epochs):
#     model.train()                                           # 학습 모드
#     running_loss = 0.0                                      # 손실값 변수 초기화
#     for batch_idx, (data, target) in enumerate(train_loader):
#         # 데이터셋에서 이미지, 경로 정보 추출
#         images, targets = data['image'], data['target']
        
#         # GPU 사용 여부 체크
#         if torch.cuda.is_available():
#             images, targets = images.cuda(), targets.cuda()

#         # 모델 예측
#         optimizer.zero_grad()                               # Optimizer 초기화
#         output = model(images)
        
#         # 손실 계산 및 역전파
#         loss = criterion(output, targets)
#         loss.backward()
#         optimizer.step()
        
#         running_loss += loss.item()                         # Loss value 누적
    
#     # epoch별 평균 손실
#     print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss / len(train_loader)}")


In [ ]:
# 수정

In [ ]:
# # 예측
# model.eval()                                                # evaluation mode run
# with torch.no_grad():                                       # no gradient
#     for data in train_loader:
#         images, targets = data['image'], data['target']
#         if torch.cuda.is_available():
#             images = images.cuda()

#         # 모델로부터 미래 경로 예측
#         predicted_paths = model(images)

#         # 미래 위치 예측 경로
#         plt.figure(figsize=(8, 6))
#         draw_trajectory(targets[0], predicted_paths[0])
#         plt.show()
